# 03 — Elim and case dispatchers

`Elim` is PLAN's pattern matcher. It takes six handlers — one per shape of value — plus the value to dispatch on, and it picks the right handler based on the value's type.


## Anatomy of `Elim`

Surface syntax:

```
(Elim p l a z m o)
```

* `p` — handler for **Pin**: applied to the pin's inner.
* `l` — handler for **Law**: applied to (name, arity, body).
* `a` — handler for **App**: applied to (head, tail).
* `z` — handler for **zero**: returned directly.
* `m` — handler for **successor n>0**: applied to n-1.
* `o` — the value to dispatch on.

The handler arms are just normal PLAN values; the `Elim` primitive picks one based on `o`'s shape and applies it.

### A simple use: zero-check

Use `Elim`'s `z` arm for the zero case and ignore everything else. (`k` is from lesson 02 — it's the constant-function pattern. Re-bind it here so this lesson stands alone.)

In [1]:
(#bind k (#law "k" (k a b) a))

bind k

In [2]:
(#bind is_zero (#law "is_zero" (is_zero n)
  (Elim 0 0 0 1 (k 0) n)))

bind is_zero

In [3]:
(is_zero 0)

1

In [4]:
(is_zero 5)

0

Reading the body: when `n` is `0`, return `1`. When `n` is a successor (any nat > 0), apply `(k 0)` to `n-1` — that discards the predecessor and yields `0`. The Pin/Law/App handlers are placeholders since `n` is always a nat here.

## `Case2`..`Case16` — small dispatchers

For matching a nat against a small set of values, BPLAN provides shortcuts. `Case2` picks branch_0 if the index is 0, fallback otherwise:

In [5]:
(Case2 0 100 999)

100

In [6]:
(Case2 1 100 999)

999

In [7]:
(Case2 50 100 999)

999

`Case3` adds a second indexed branch:

In [8]:
(Case3 0 "first" "second" "fallback")

500153084262

In [9]:
(Case3 1 "first" "second" "fallback")

110425477965171

In [10]:
(Case3 2 "first" "second" "fallback")

7738135660106375526

(The strings are encoded as nats — that's why the cell shows numbers. Plan Asm doesn't have a separate string type; `"foo"` is shorthand for the nat whose little-endian bytes decode as `"foo"`.)

## Encoding booleans with laws

Church-style booleans use two-arg laws — `T a b = a`, `F a b = b`. Then `(b x y)` is "if b then x else y".

In [11]:
(#bind T (#pin (#law "T" (T a b) a)))

bind T

In [12]:
(#bind F (#pin (#law "F" (F a b) b)))

bind F

In [13]:
(T 100 999)

100

In [14]:
(F 100 999)

999

Logical AND: `And p q = (p q F)` — if `p` is true, return `q`; else return `F`.

In [15]:
(#bind And (#pin (#law "And" (And p q) (p q F))))

bind And

In [16]:
(And T T 100 999)

100

In [17]:
(And T F 100 999)

999

## What's next

Lesson 04 introduces **recursion**. The signature's `self` name lets a law refer to itself, which is how you build factorial, fibonacci, and the rest of the usual recursive shapes.